<a href="https://colab.research.google.com/github/Bhoomika253/1MS25IS026_Assignment2/blob/main/Copy_of_agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [9]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [3]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
GAMES = {

   "cricket": (
        "Cricket is a bat-and-ball team sport played between two teams of eleven players."
        "The game is played on a large oval field with a rectangular pitch in the center."
        "The main objective is to score more runs than the opposing team."
        "Players can bat, bowl, field, or perform wicketkeeping during a match."
    ),

    "football": (
        "Football is a team sport played between two teams of eleven players."
        "The game is played on a rectangular field with a goal at each end."
        "Players try to score by getting the ball into the opponent's goal."
        "The goalkeeper is the player responsible for protecting the goal and can use their hands within the penalty area."
    ),

    "badminton": (
        "Badminton is a racket sport played with a lightweight shuttlecock."
        "It can be played between two players in singles or four players in doubles."
        "Players hit the shuttlecock over a net and try to make it land inside the opponent's court."
        "The game requires quick reflexes, agility, coordination, and accurate racket shots."
    ),

    "chess": (
        "Chess is a two-player strategy game played on a board containing sixty-four squares."
        "Each player starts with sixteen pieces, including a king, queen, rooks, bishops, knights, and pawns."
        "Players move their pieces according to specific rules and try to checkmate the opponent's king."
        "The game requires strategic planning, logical thinking, calculation, and careful decision-making."
    )

}

In [4]:
# @title Define a tool
def search_GAMES(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in GAMES.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in str(content).lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant information found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_GAMES("cricket"))


[Source: cricket]
Cricket is a bat-and-ball team sport played between two teams of eleven players.The game is played on a large oval field with a rectangular pitch in the center.The main objective is to score more runs than the opposing team.Players can bat, bowl, field, or perform wicketkeeping during a match.


In [10]:
# @title Ask gemini to search for something
import os
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="gemini-3.5-flash-lite",
    tools=[search_GAMES],
    system_instruction=(
        """
    You are a Multi-Sport AI Assistant specializing in cricket,
football, chess, and badminton.

When answering questions, use the search_GAMES tool
to retrieve relevant information from the provided documents.

RULES:
1. Answer only using information explicitly supported
   by the retrieved documents.
2. Check whether the retrieved content actually answers
   the user's specific question.
3. Do not use outside knowledge, guesses, jokes, puns,
   or assumptions.
4. A document mentioning a sport does not mean it answers
   the question.
5. If the documents do not contain sufficient information,
   respond exactly:
   "I couldn't find the answer in the provided documents."
6. Never invent an answer.
    """
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-3.5-flash-lite


In [6]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_GAMES(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What is the best game to play when more number of players are there?")

Question: What is the best game to play when more number of players are there?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'best game more number of players'})
  Result preview: [Source: cricket]
Cricket is a bat-and-ball team sport played between two teams of eleven players.Th...

Final answer:
When you have a larger number of players, team sports like **Cricket** and **Football** are among the best choices. Both games involve two teams of **eleven players** each (making a total of 22 players on the field), providing great opportunities for teamwork, active participation, and large-scale competition.


In [7]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "What are the rules of chess?",
    "How does scoring works in badminton?",
    "What is the role of an umpire in cricket?",
    "How many players are on a football team?",
    # This question is not answered by the database
    "What is the secret password of the chess champion in my documents?"
    # This question is part of the general knowledge of the model.
    "In which year India won world cup for the first time?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: What are the rules of chess?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'rules of chess'})
  Result preview: [Source: chess]
Chess is a two-player strategy game played on a board containing sixty-four squares....

Final answer:
Based on the provided documents, chess is a two-player strategy game played on a board with sixty-four squares where each player starts with sixteen pieces (including a king, queen, rooks, bishops, knights, and pawns). Players move their pieces according to specific rules and try to checkmate the opponent's king.

Question: How does scoring works in badminton?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'badminton scoring system points games match'})
  Result preview: [Source: cricket]
Cricket is a bat-and-ball team sport played between two teams of eleven players.Th...
Step 2: model calls search_documents({'query': 'badminton scoring rules'})


In [8]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)